
# 1. INSTALL DEPENDENCIES

In [1]:

!pip install --q transformers datasets accelerate evaluate

In [2]:
import os
import torch
from datasets import load_dataset
from transformers import (
    GPT2Config,
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    pipeline
)

# 2. LOAD DATASET


In [3]:

print("Reading raw file from local disk...")
with open("/content/TinyStoriesV2-GPT4-valid.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# Separate the file by the actual story boundary token
stories = raw_text.split("<|endoftext|>")
# Strip whitespaces and filter out empty strings
stories = [story.strip() for story in stories if story.strip()]

print(f"Total authentic stories found: {len(stories)}")

# Convert the raw list of strings into a Hugging Face Dataset format
from datasets import Dataset
full_dataset = Dataset.from_dict({"text": stories})

Reading raw file from local disk...
Total authentic stories found: 27630


In [4]:
full_dataset['text'][10]

"Once upon a time, in a small house, there lived a little boy named Tim. Tim was an intelligent boy who loved to learn new things. One day, he found a big book about ghosts. He read the book and learned that ghosts could be nice or scary.\nOne day, Tim met a friendly ghost named Gigi. Gigi was lost and needed help. Tim wanted to show Gigi that he was a good friend. So, he picked up Gigi's favorite toy, a round ball. But, as he was playing with the ball, he accidentally let it drop. The ball broke into pieces. Tim felt sad and said sorry to Gigi.\nGigi smiled and told Tim that it was okay. She said that everyone makes mistakes. The important thing is to say sorry and learn from them. Tim learned that being a good friend means helping each other and being kind. And from that day on, Tim and Gigi became the best of friends."

 # SPLIT DATASET INTO TRAIN AND TEST

In [5]:
# Safe, sequential 80/20 split without shuffling paragraphs
total_records = len(full_dataset)
split_index = int(total_records * 0.8)

train_dataset = full_dataset.select(range(0, split_index))
test_dataset = full_dataset.select(range(split_index, total_records))

print(f"Train set (first 80%): {len(train_dataset)} entire stories")
print(f"Test set (remaining 20%): {len(test_dataset)} entire stories")

Train set (first 80%): 22104 entire stories
Test set (remaining 20%): 5526 entire stories


# 4. TOKENIZATION


In [34]:
context_length = 256
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(element):
    outputs = tokenizer(
        element["text"],
        truncation=True,
        max_length=context_length,
        return_overflowing_tokens=True,
        return_length=True,
    )
    input_batch = []
    for length, input_ids in zip(outputs["length"], outputs["input_ids"]):
        if length == context_length:
            input_batch.append(input_ids)
    return {"input_ids": input_batch}

In [7]:
print("Tokenizing local stories sequentially...")
tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_test = test_dataset.map(tokenize_function, batched=True, remove_columns=test_dataset.column_names)

Tokenizing local stories sequentially...


Map:   0%|          | 0/22104 [00:00<?, ? examples/s]

Map:   0%|          | 0/5526 [00:00<?, ? examples/s]

 # 5. CREATE THE MODEL

In [21]:

config = GPT2Config(
    vocab_size=len(tokenizer),
    n_positions=context_length,
    n_ctx=context_length,
    n_embd=288,
    n_layer=6,
    n_head=6,
)

model = GPT2LMHeadModel(config)
model_size = sum(t.numel() for t in model.parameters())
print(f"Total Model Parameters: {model_size/1e6:.2f}M")

Total Model Parameters: 20.54M


In [22]:
model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 288)
    (wpe): Embedding(256, 288)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((288,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=864, nx=288)
          (c_proj): Conv1D(nf=288, nx=288)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((288,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=1152, nx=288)
          (c_proj): Conv1D(nf=288, nx=1152)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((288,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=288, out_features=50257, bias=False)
)

## 5.5 TEST UNTRAINED GENERATION (ON CPU)

In [10]:
# Configure special tokens safely
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id

# Run pipeline explicitly on CPU (device="cpu" or -1)
untrained_generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device="cpu")
prompt = "Once upon a time, a little puppy found a"

untrained_res = untrained_generator(
    prompt,
    max_new_tokens=40,
    num_return_sequences=1,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id
)

print("Prompt:", prompt)
print("\n--- Untrained Model Output (Junk) ---")
print(untrained_res[0]['generated_text'])
print("-" * 38)

# Clean up the pipeline object to prep for GPU training
del untrained_generator
model.config.use_cache = False

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'eos_token_id', 'do_sample', 'temperature', 'num_return_sequences', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Testing Untrained Model Generation ---


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Prompt: Once upon a time, a little puppy found a

--- Untrained Model Output (Junk) ---
Once upon a time, a little puppy found a Olive discourage monthlyoutheast Alexander incentive Argentulptrulylus Velvet GumCmd Preselloes landfill analys spinachalties freight Testing negligencebased bossackers______¨ FIGHTdial extensions Ples taxpayerenicintrodu regeneration stationoakhyper central
--------------------------------------


# 6. CONFIG TRAINER

In [35]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [12]:
os.environ['TENSORBOARD_LOGGING_DIR'] = '/content/logs'

In [23]:
training_args = TrainingArguments(
    output_dir="./pretrain",
    num_train_epochs=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="steps",            # Note: evaluation_strategy was also updated to eval_strategy
    eval_steps=200,
    logging_steps=50,
    save_steps=400,
    learning_rate=5e-4,
    weight_decay=0.01,
    fp16=True,
    report_to="none"
)

In [24]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
)

# 7. RUN PRETRAINING

In [25]:
print("Starting pretraining on complete, unbroken stories...")
trainer.train()

Starting pretraining on complete, unbroken stories...


Step,Training Loss,Validation Loss
200,3.835945,3.797640
400,3.271664,3.328779
600,2.940533,3.059824
800,2.769586,2.944434
890,2.726207,2.927405


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=890, training_loss=3.4754746940698515, metrics={'train_runtime': 250.6894, 'train_samples_per_second': 112.41, 'train_steps_per_second': 3.55, 'total_flos': 259490803875840.0, 'train_loss': 3.4754746940698515, 'epoch': 10.0})

# 8. SAVE, WIPE GPU, & INFERENCE ON CPU


In [26]:

import gc
import torch

print("\nSaving trained model and tokenizer to disk...")
trainer.save_model("./tiny_stories_final")
tokenizer.save_pretrained("./tiny_stories_final")

print("Flushing GPU memory...")
# Delete the active GPU-bound model and trainer
del model
del trainer
# Force Python garbage collection and CUDA cache clearing
gc.collect()
torch.cuda.empty_cache()


Saving trained model and tokenizer to disk...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Flushing GPU memory...


# 9. GENERATE STORIES

In [33]:
print("Loading saved model onto CPU for safe inference...")
# Load directly from the saved directory onto the CPU
cpu_model = GPT2LMHeadModel.from_pretrained("./tiny_stories_final").to("cpu")

# Re-apply generation configs to prevent index errors
cpu_model.config.pad_token_id = tokenizer.pad_token_id
cpu_model.config.eos_token_id = tokenizer.eos_token_id
cpu_model.generation_config.pad_token_id = tokenizer.pad_token_id
cpu_model.generation_config.eos_token_id = tokenizer.eos_token_id

print("Testing pre-trained model inference (CPU)...")
generator = pipeline("text-generation", model=cpu_model, tokenizer=tokenizer, device="cpu")
prompt = "Once upon a time, a little puppy found a"

res = generator(
    prompt,
    max_new_tokens=100,
    num_return_sequences=1,
    temperature=0.1,
    do_sample=False,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id
)

print("\n--- Final Pre-Trained Model Output ---")
print(res[0]['generated_text'])

Loading saved model onto CPU for safe inference...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Testing pre-trained model inference (CPU)...

--- Final Pre-Trained Model Output ---
Once upon a time, a little puppy found a little girl named Lucy. She was very happy to play with her friends. She loved to play with her mom, but she was very happy.
One day, she saw a big box in the box. She wanted to play with her mom. She wanted to play with her mom. She said, "I want to play with you."
Her mom said, "I can I want to play with you." She said, "I want to play with you." She said, "I
